In [ ]:
import io
import os
import subprocess
import pandas as pd
import firecloud.api as fapi

# =============================== CONFIG ======================================
DATASET_NAME = "ARIC"          # cohort name, e.g. "GeneSTAR", "WHI", "MESA"...
SUB_NAME = "t2d"                   # trait
ENTITY_TYPE = "pgs_scoring_file"   # data table name (assumed same across workspaces)
BUCKET_SUBFOLDER = "pgs_combined_summaries"  # subfolder under the workspace bucket
MATCH_STATUS_FILTER = "matched"    # status used for the flipped/not-flipped split
# ==============================================================================

SUMMARY_FILENAME = f"{DATASET_NAME}_summary.csv"
OUTPUT_COMBINED = f"{DATASET_NAME}_{SUB_NAME}_summary_combined.csv"
MATCH_RATE_FILENAME = f"{DATASET_NAME}_{SUB_NAME}_match_rate.csv"


In [ ]:
# ---- 1. Pull the pgs_scoring_file data table directly from the workspace --
# Terra/AnVIL notebook runtimes set these env vars automatically for the
# workspace the notebook is running in.
WORKSPACE_NAMESPACE = os.environ["WORKSPACE_NAMESPACE"]
WORKSPACE_NAME = os.environ["WORKSPACE_NAME"]
ENTITY_TYPE = f"pgs_scoring_file_{SUB_NAME}"  # pgs_scoring_file

response = fapi.get_entities_tsv(
    WORKSPACE_NAMESPACE, WORKSPACE_NAME, ENTITY_TYPE, model="flexible"
)
response.raise_for_status()
 
scoring_table = pd.read_csv(io.StringIO(response.text), sep="\t")
print(f"Pulled {len(scoring_table)} rows from data table '{ENTITY_TYPE}' "
      f"in {WORKSPACE_NAMESPACE}/{WORKSPACE_NAME}")


In [ ]:
# ---- 2. Helper to read a CSV directly from a gs:// path --------------------
def read_gs_csv(gs_path: str) -> pd.DataFrame:
    raw = subprocess.run(
        ["gsutil", "cat", gs_path],
        check=True,
        capture_output=True,
    ).stdout
    return pd.read_csv(io.BytesIO(raw))
 

In [ ]:
# ---- 3. Loop over every row, fetch its summary file, and stack ------------
combined_frames = []
failures = []
 
for _, row in scoring_table.iterrows():
    pgs_id = row["pgs_model_id"]
    gs_path = row["match_summary"]
 
    try:
        df = read_gs_csv(gs_path)
    except subprocess.CalledProcessError as e:
        failures.append((pgs_id, gs_path, e.stderr.decode(errors="replace")))
        continue
 
    # Sanity check: the file's own 'accession' column should match pgs_model_id
    if "accession" in df.columns and not (df["accession"] == pgs_id).all():
        print(f"WARNING: accession mismatch in file for {pgs_id}")
 
    df["pgs_scoring_file_id"] = row[f"entity:pgs_scoring_file_{SUB_NAME}_id"]
    combined_frames.append(df)
 
if failures:
    print(f"\n{len(failures)} file(s) failed to download:")
    for pgs_id, path, err in failures:
        print(f"  {pgs_id}: {path}\n    {err.strip()}")


In [ ]:
# ---- 4. Concatenate and save ------------------------------------------------
combined = pd.concat(combined_frames, ignore_index=True)
print(f"\nCombined {len(combined_frames)} files -> {combined.shape[0]} rows, "
      f"{combined['accession'].nunique()} unique PGS accessions.")
 
combined.to_csv(OUTPUT_COMBINED, index=False)
print(f"Saved combined table to {OUTPUT_COMBINED}")
 

In [ ]:
# ---- 5. (Optional) quick sanity summary ------------------------------------
print("\nRow counts by match_status across all PGS scores:")
print(combined.groupby("match_status")["count"].sum())

# Total variant count per PGS score (sum of 'count' across all its
# match_status rows), added as a column so each row shows its share of the
# whole for that accession.
combined["total_count"] = combined.groupby("accession")["count"].transform("sum")


In [ ]:
# ---- 6. Filter by match_status / match_flipped -----------------------------
# match_flipped is blank ("") for unmatched rows, so coerce to a real bool
# before filtering rather than comparing against the string "true"/"false".
combined["match_flipped_bool"] = (
    combined["match_flipped"].astype(str).str.lower().map({"true": True, "false": False})
)
 
def filter_summary(df, match_status=None, match_flipped=None):
    """
    match_status: str or list of str, e.g. "matched" or ["matched", "excluded"]
    match_flipped: True, False, or None (None = don't filter on this column)
    """
    out = df.copy()
    if match_status is not None:
        statuses = [match_status] if isinstance(match_status, str) else match_status
        out = out[out["match_status"].isin(statuses)]
    if match_flipped is not None:
        out = out[out["match_flipped_bool"] == match_flipped]
    return out
 
# Example: matched variants NOT flipped (i.e. the "clean" matches)
matched_not_flipped = filter_summary(combined, match_status="matched", match_flipped=False)
print(f"matched + not flipped: {len(matched_not_flipped)} rows, "
      f"{matched_not_flipped['count'].sum()} total variants")
 
match_rate = matched_not_flipped[["dataset", "accession", "total_count", "count", "percent", "pgs_scoring_file_id"]]
match_rate.columns = ["cohort", "pgs_model_id", "total_count", "count", "percent", "pgs_scoring_file_id"]
match_rate.to_csv(MATCH_RATE_FILENAME, index=False)


In [ ]:
# ---- 7. Copy outputs back to the Terra workspace bucket --------------------
# Terra notebook runtimes set WORKSPACE_BUCKET automatically (e.g.
# "gs://fc-secure-xxxxxxxx-xxxx-...").
WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
DEST_PREFIX = f"{WORKSPACE_BUCKET}/pgs_comparison_results/"  # change subfolder as you like
 
output_files = [
    MATCH_RATE_FILENAME,
    OUTPUT_COMBINED,
]
 
for f in output_files:
    subprocess.run(["gsutil", "cp", f, DEST_PREFIX], check=True)
    print(f"Uploaded {f} -> {DEST_PREFIX}{f}")
